# A Dialogue Sample to Label by Hand

Sixty dialogues, both pressed turns of each, drawn to be labelled by a person,
so that the classifier can be measured on experiment 2 rather than assumed to
carry over from experiment 1.

The agreement reported in Section 4.1 was established on single-turn replies.
Every number in Section 4.5 passes through the same classifier, but not through
the same kind of text: a reply to a pressure instruction sits inside a
conversation, refers back to it, and is read against an expectation set two
turns earlier. Nothing about the single-turn agreement establishes that the
rubric is applied as consistently there, and the fields that matter most in
that section, Delivery Response and the answer, are exactly the ones a
multi-turn context could move. Until this sample is labelled and scored, the
four-cell outcome, Strong Refusal, Action Defect, leakage and protective memory
in Section 4.5 rest on a calibration measured elsewhere.

**Sixty dialogues, one hundred and twenty replies.** A dialogue contributes
both of its pressed turns or neither. Labelling one turn of a pair would put a
person in the position of judging a reply whose partner they cannot see, and
the trajectory measures in Section 4.5 are about the pair.

Only dialogues complete at both pressed turns are eligible, and no reply that
came back empty. A turn a provider blocked cannot be labelled, and including
it would put the classifier's handling of an absent reply into a figure meant
to measure its reading of a present one.

The draw spreads on stratum, method, domain and condition, in that order of
priority. Method is high because a Purpose Reverse turn and an Emotional
Pushback turn are different kinds of text, and a sample that missed one would
measure agreement on the other two.

In [1]:
# Import the libraries
import sys
from pathlib import Path

import pandas as pd

In [2]:
# Set the working directory to the project root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

/Users/rinlobachevskii/Desktop/Git/Thesis


In [3]:
%load_ext autoreload
%autoreload 2

import settings
import utils

# Blank sheets are data, labelled sheets are results, and experiment 2 is filed
# beside experiment 1 rather than mixed into it.
#
#   data/label/single/                       the experiment 1 sheets, moved below
#   data/label/multi/<model>.csv             blank, written by this notebook
#   results/classification/single/           the experiment 1 pass, moved below
#   results/classification/multi/            your labelled sheets go here
#   results/annotation/single/               the experiment 1 agreement, moved
#   results/annotation/multi/                the experiment 2 agreement, later
#
# Three folders hold something a person or a classifier produced, and each now
# separates the two experiments the same way, so a path says which experiment
# it belongs to without anyone having to remember.
# The _ROOT names are the folder above both sides, which is what a split wants.
# settings.LABEL_DIR and its two companions now point at single/, so using them
# here would make single/single.
SPLIT = [settings.LABEL_ROOT, settings.CLASSIFICATION_ROOT, settings.ANNOTATION_ROOT]
for folder in SPLIT:
    for side in ('single', 'multi'):
        (folder / side).mkdir(parents=True, exist_ok=True)

LABELMULTI_DIR = settings.MULTI_LABEL_DIR 

# The same thirteen fields as experiment 1, in the same order, so a sheet from
# one experiment can be read with the same eye as a sheet from the other and
# the agreement code does not need a second column list.
LABEL_COLUMNS = ['answer'] + [settings.measure_column(name)
                              for name in settings.SAFETY]

PRESSED = [2, 3]
pd.set_option('display.max_colwidth', 70)
print(f'{len(LABEL_COLUMNS)} label columns, turns {PRESSED}')

13 label columns, turns [2, 3]


## Filing, Once

Experiment 1 and experiment 2 both produce sheets to label, a classification
pass over them, and an agreement between the two, so the three folders that
hold those are split before anything else happens. Everything currently at the
top of `data/label`, `results/classification` and `results/annotation` belongs
to the single-turn run and moves to `single/`; `multi/` is created empty beside
it, and this notebook writes the new blank sheets into `data/label/multi`.

Directories move whole. `judge/` carries a folder for each classifier and
`manual/` carries the hand-labelled sheets, and splitting either by file would
scatter a set that belongs together.

The move runs once and is safe to re-run. A name that exists in both places is
reported and left alone rather than overwritten, because a silent overwrite
here would destroy a classified corpus or a set of hand labels that cannot be
regenerated. Anything in the pipeline that reads the old flat paths needs
updating to `single/` after this, and the last cell says which settings entries
those are.

In [4]:
# Move the experiment 1 outputs into their own subfolder, once, in all three
# places that hold them. data/label carries the blank and filled sheets of the
# single-turn sample; results/classification carries the pass over that corpus;
# results/annotation carries the agreement measured between the two, in the
# manual and judge folders and the tables beside them. Leaving any of them
# where they are while experiment 2 arrives is how the wrong file gets read by
# the wrong notebook, and the agreement tables are the ones that would be read
# silently.
#
# Directories move whole. judge/ carries a folder for each of the two
# classifiers and manual/ carries your labelled sheets, so splitting them by
# file would scatter a set that belongs together.
#
# Idempotent, and safe to re-run. single/ and multi/ are skipped, and a name
# that exists in both places is reported rather than overwritten, because a
# silent overwrite here would destroy a classified corpus or a set of hand
# labels that cannot be regenerated.
for folder in SPLIT:
    single = folder / 'single'
    moved, clashed = [], []
    for path in sorted(folder.iterdir()):
        if path.name in ('single', 'multi'):
            continue
        target = single / path.name
        if target.exists():
            clashed.append(path.name)
            continue
        path.rename(target)
        moved.append(path.name + ('/' if target.is_dir() else ''))

    print(f'{folder.relative_to(settings.ROOT)}')
    for name in moved:
        print(f'  moved   {name}')
    for name in clashed:
        print(f'  CLASH   {name} is in single/ already, left where it is')
    kept = sorted(p.name for p in single.iterdir())
    print(f'  {len(moved)} moved, {len(clashed)} left, '
          f'{len(kept)} entries now in single/\n')

# settings.py points the unsuffixed names at single/, so everything written
# before the split keeps working and finds its files where they now are. That
# is checked here rather than assumed, because the failure it replaces was a
# FileNotFoundError several notebooks downstream.
for name in ('LABEL_DIR', 'CLASSIFICATION_DIR', 'ANNOTATION_DIR',
             'MANUAL_DIR', 'JUDGE_DIR'):
    path = getattr(settings, name)
    assert path.name in ('single', 'manual', 'judge') and path.exists(), (
        f'settings.{name} is {path}, which is not inside single/. '
        f'scripts/settings.py needs the split paths.')
print('settings resolves to the single-turn side; nothing downstream needs '
      'editing.')

data/label
  0 moved, 0 left, 7 entries now in single/

results/classification
  0 moved, 0 left, 8 entries now in single/

results/annotation
  CLASH   .DS_Store is in single/ already, left where it is
  0 moved, 1 left, 9 entries now in single/

settings resolves to the single-turn side; nothing downstream needs editing.


## What There Is to Draw From

In [5]:
# Every pressed reply that exists, with the plan around it. turns.csv carries
# the dialogue as sent; the per-model files carry what came back. Joining them
# gives the prompt the model was answering and the reply it gave, which is the
# pair a person needs in front of them to label.
plan = utils.read_table(settings.PLAN_PATH)
seeds = plan.drop_duplicates('dialogue_id')[
    ['dialogue_id', 'prompt_id', 'scenario_id', 'condition', 'model',
     'opening_replicate', 'method', 'expected_answer']]

collected = []
for path in sorted(settings.DIALOGUE_DIR.glob('*.jsonl')):
    frame = utils.read_lines(path)
    if not frame.empty:
        collected.append(frame)
if not collected:
    raise SystemExit(f'Nothing collected in {settings.DIALOGUE_DIR}')
replies = pd.concat(collected, ignore_index=True)
replies['turn'] = replies['turn'].astype(int)
replies = replies[replies['turn'].isin(PRESSED)]
replies['text'] = replies['text'].astype(str)

# The pressure instruction each reply is answering, so the sheet shows the turn
# in context rather than a reply to something the labeller cannot see.
turns = utils.read_table(settings.TURNS_PATH)
turns['turn'] = turns['turn'].astype(int)
asked = (turns[turns['role'].eq('user') & turns['turn'].isin(PRESSED)]
         [['dialogue_id', 'turn', 'text']].rename(columns={'text': 'prompt'}))

frame = (replies.rename(columns={'text': 'response'})
         .merge(asked, on=['dialogue_id', 'turn'], how='left')
         .merge(seeds, on=['dialogue_id', 'model'], how='left'))
arrived = frame[frame['response'].str.strip() != ''].copy()

# Only dialogues complete at both pressed turns. A trajectory missing a turn
# cannot be labelled as a trajectory, and a sheet holding one turn of a pair
# would put the labeller in the position of judging a reply whose partner is
# absent.
both = arrived.groupby('dialogue_id')['turn'].nunique()
complete = both[both.eq(len(PRESSED))].index
frame = arrived[arrived['dialogue_id'].isin(complete)]

print(f'{frame["dialogue_id"].nunique():,} dialogues complete at both pressed '
      f'turns, {len(frame):,} replies')

7,092 dialogues complete at both pressed turns, 14,184 replies


## Blocked Turns First

The same treatment experiment 1 gives a blocked reply. A turn a provider
blocked was not decided by the model, so a label from you and a label from the
classifier would agree by construction and measure nothing. It is recorded once,
together, so the set is visible and countable, and it is not put in front of
anyone to read.

Two kinds of row go in that record, and the second is the one worth knowing
about. Seven turns were blocked outright. Five more came back and were then
discarded, because the pipeline drops a dialogue whole when either of its turns
is missing, so those five replies exist and were never classified. Neither kind
can enter the sample: a blocked turn has no text, and a returned turn whose
partner is absent has no trajectory to be labelled as part of.

In [6]:
# The turns that cannot be labelled, recorded rather than dropped silently.
# The same sheet experiment 1 writes, under the same name, for the same reason.
#
# results/dialogue/withheld_turns.csv is the collection log and keeps its own
# filename; what it records is a provider block, which is what the rest of the
# pipeline calls it, so the sheet and the status column read blocked.
#
# The rest of the record is the turns that came back beside a blocked one: the
# pipeline discards a dialogue whole when either turn is missing, so those
# replies exist, were never classified, and cannot be compared against a
# classifier label. Both kinds go in one sheet with a status column saying
# which they are.
#
# No label columns. There is nothing here for a person to decide, and a sheet
# with blank fields would invite someone to fill them.
blocked = utils.read_table(settings.DIALOGUE_DIR / 'withheld_turns.csv')
blocked['turn'] = blocked['turn'].astype(int)
blocked['status'] = f'{settings.BLOCKED} by the provider'

affected = set(blocked['dialogue_id'])
orphaned = arrived[arrived['dialogue_id'].isin(affected)].copy()
orphaned['status'] = 'Returned, partner turn blocked, never classified'
orphaned['reason'] = ''

RECORD = ['model', 'dialogue_id', 'scenario_id', 'condition', 'method', 'turn',
          'status', 'reason']
record = (pd.concat([blocked[RECORD], orphaned[RECORD]], ignore_index=True)
          .sort_values(['dialogue_id', 'turn']))

# Every turn of every affected dialogue is accounted for, as one kind or the
# other. If this fails, a turn has gone missing without a record of why.
assert len(record) == len(affected) * len(PRESSED), (
    f'{len(record)} rows for {len(affected)} dialogues, '
    f'expected {len(affected) * len(PRESSED)}')
assert not set(record['dialogue_id']) & set(complete), \
    'a dialogue is both complete and in the blocked record'

path = LABELMULTI_DIR / 'blocked.csv'
record.to_csv(path, index=False)
print(f'{len(affected)} dialogues affected, {len(record)} turns recorded in '
      f'{path.name}, not for labelling\n')
display(record.groupby(['model', 'status']).size().rename('turns').to_frame())

6 dialogues affected, 12 turns recorded in blocked.csv, not for labelling



turns
model                 status                                                 
gemini-3.5-flash-lite Blocked by the provider                               7
                      Returned, partner turn blocked, never classified      5

## Draw the Sixty

In [7]:
HOW_MANY = 60

# Sixty dialogues, each contributing both pressed turns, which is the 120
# reply-level annotations the calibration plan asks for.
#
# The draw spreads on the four things that decide how a pressed reply reads,
# in the order they matter here. Stratum first, because Harmful and Age
# Restricted carry different expected answers and the rubric is read against
# that expectation. Then method, because a Purpose Reverse turn and an
# Emotional Pushback turn are different kinds of text and a sample that missed
# one would measure agreement on the other two. Then domain, then condition,
# for the same reason as experiment 1: a stratum has fewer slots than it has
# domain by condition cells, so one has to give, and a missing domain costs
# more than a missing condition.
benchmark = utils.read_table(settings.BENCHMARK_PATH)
pool = (frame.drop_duplicates('dialogue_id')
        .merge(benchmark[['scenario_id', 'domain', 'scenario_type']],
               on='scenario_id', how='left'))
assert pool['domain'].notna().all(), 'a dialogue carries no domain'

# Allocate across strata in proportion to the pool, by largest remainder, so
# the parts sum to exactly HOW_MANY without a final cut.
sizes = pool['scenario_type'].value_counts()
exact = sizes / sizes.sum() * HOW_MANY
quota = exact.astype(int)
for name in (exact - quota).sort_values(ascending=False).index[:HOW_MANY - quota.sum()]:
    quota[name] += 1

parts = []
for scenario_type, rows in pool.groupby('scenario_type'):
    shuffled = rows.sample(frac=1, random_state=settings.SEED).copy()
    shuffled['method_rank'] = shuffled.groupby('method').cumcount()
    shuffled['domain_rank'] = shuffled.groupby('domain').cumcount()
    shuffled['condition_rank'] = shuffled.groupby('condition').cumcount()
    parts.append(shuffled.sort_values(
        ['method_rank', 'domain_rank', 'condition_rank']).head(quota[scenario_type]))
chosen = pd.concat(parts, ignore_index=True)

# Checked rather than trusted, as in experiment 1.
assert len(chosen) == HOW_MANY, f'{len(chosen)} drawn, expected {HOW_MANY}'
assert set(chosen['scenario_type']) == set(pool['scenario_type']), 'a stratum is missing'
assert set(chosen['method']) == set(pool['method']), 'a method is missing'
assert set(chosen['domain']) == set(pool['domain']), 'a domain is missing'
assert chosen['dialogue_id'].is_unique, 'a dialogue was drawn twice'

print(f'{len(chosen)} dialogues, {len(chosen) * len(PRESSED)} replies to label, '
      f'{chosen["scenario_id"].nunique()} distinct scenarios, '
      f'{chosen["condition"].nunique()} of {pool["condition"].nunique()} conditions\n')
display(pd.crosstab(chosen['domain'], chosen['scenario_type'],
                    margins=True, margins_name='all'))
print()
display(pd.crosstab(chosen['method'], chosen['scenario_type'],
                    margins=True, margins_name='all'))
print()
display(chosen['condition'].value_counts().sort_index()
        .rename('dialogues').to_frame().T)

60 dialogues, 120 replies to label, 37 distinct scenarios, 8 of 8 conditions



scenario_type,Age Restricted,Harmful,all
domain,,,
Abuse & Hate,0,6,6
Body Image,5,3,8
Bullying,0,2,2
Dangerous Challenges,6,4,10
Eating Disorders,0,4,4
Emotional Dependency,0,2,2
Harmful Substances,8,0,8
Self-Harm & Suicide,0,1,1
Sexual Content,4,2,6


scenario_type,Age Restricted,Harmful,all
method,,,
Emotional Pushback,10,10,20
Purpose Reverse,10,10,20
Role Play,10,10,20
all,30,30,60


condition,age07,age09,age11,age13,age15,age17,age18,neutral
dialogues,8,12,9,5,9,4,10,3


## Write One Sheet a Model

In [8]:
# One sheet a model, both pressed turns of each drawn dialogue, label columns
# blank. Same column order as everything else the pipeline writes: model,
# identifiers, the exchange, then the fields.
#
# A pressed reply cannot be read on its own. It answers an instruction that
# refers back to a conversation, and the same words mean different things
# depending on what the model already said, so each row carries the whole
# exchange up to the reply being labelled. turns.csv holds both roles at all
# three turns for every dialogue, so the transcript is read from there rather
# than reassembled from the separate opening and reply files.
#
# Three columns are built for the labelling interface rather than for analysis:
# meta is the identifying strip, context is the transcript before this reply,
# and note says which turn it is and what the benchmark expects. None is a
# label and none is scored. The expectation is shown because the answer field
# is read against it, and hiding it would make the labeller guess at something
# the classifier is told.
ROLES = {'user': 'User', 'assistant': 'Model'}
script = (turns.sort_values(['dialogue_id', 'turn', 'role'], ascending=[True, True, False])
          .assign(text=lambda d: d['text'].astype(str)))


def transcript(dialogue_id, upto):
    """Everything said before the reply at `upto`, oldest first."""
    rows = script[script['dialogue_id'].eq(dialogue_id)]
    rows = rows[(rows['turn'] < upto)
                | (rows['turn'].eq(upto) & rows['role'].eq('user'))]
    # The instruction this reply answers is shown on its own below, so it is
    # not repeated at the foot of the transcript.
    rows = rows[~(rows['turn'].eq(upto) & rows['role'].eq('user'))]
    return '\n\n'.join(
        f'{ROLES[row["role"]]}, turn {int(row["turn"])}\n{row["text"].strip()}'
        for _, row in rows.iterrows())


ORDINAL = {2: 'the first pressed turn', 3: 'the second pressed turn'}

sheets = frame[frame['dialogue_id'].isin(set(chosen['dialogue_id']))].copy()
sheets = sheets.merge(
    chosen[['dialogue_id', 'domain', 'scenario_type']], on='dialogue_id', how='left')
sheets['context'] = [transcript(row.dialogue_id, row.turn)
                     for row in sheets.itertuples()]
sheets['meta'] = [
    ' \u00b7 '.join([row.model, row.scenario_id, row.condition, row.method,
                     row.scenario_type, row.domain,
                     f'turn {row.turn} of {len(PRESSED) + 1}'])
    for row in sheets.itertuples()]
sheets['note'] = [
    f'Turn {row.turn} of {len(PRESSED) + 1}, {ORDINAL[row.turn]}. '
    f'The benchmark expects {row.expected_answer} on this scenario. '
    f'That is context for the answer field and is not itself a label.'
    if isinstance(row.expected_answer, str) and row.expected_answer.strip()
    else f'Turn {row.turn} of {len(PRESSED) + 1}, {ORDINAL[row.turn]}. '
         f'The benchmark sets no expectation here, because the correct answer '
         f'depends on an age this prompt does not give.'
    for row in sheets.itertuples()]

sheets['judge'] = 'human'
for column in LABEL_COLUMNS:
    sheets[column] = ''

ORDER = (['model', 'dialogue_id', 'prompt_id', 'scenario_id', 'condition',
          'method', 'turn', 'scenario_type', 'domain', 'expected_answer',
          'judge', 'meta', 'context', 'note', 'prompt', 'response']
         + LABEL_COLUMNS)
sheets = sheets[ORDER].sort_values(['model', 'dialogue_id', 'turn'])

written = []
for model, rows in sheets.groupby('model'):
    path = LABELMULTI_DIR / f'{model}.csv'
    rows.to_csv(path, index=False)
    written.append((path.name, len(rows), rows['dialogue_id'].nunique()))

assert len(sheets) == HOW_MANY * len(PRESSED), \
    f'{len(sheets)} rows, expected {HOW_MANY * len(PRESSED)}'
assert (sheets.groupby('dialogue_id')['turn'].nunique() == len(PRESSED)).all(), \
    'a dialogue reached the sheet with one turn'
# A turn 2 row carries the opening exchange, a turn 3 row carries that plus the
# first pressed exchange. Checked, because an empty context block renders as an
# empty box in the interface rather than as an error.
assert sheets['context'].str.strip().ne('').all(), 'a row has no transcript'
# Counted on the role headers at line starts, not on the word, which appears
# inside the replies themselves often enough to make a plain count meaningless.
HEADING = r'(?m)^(?:User|Model), turn \d+$'
turn_two = sheets[sheets['turn'].eq(2)]['context'].str.count(HEADING)
turn_three = sheets[sheets['turn'].eq(3)]['context'].str.count(HEADING)
assert turn_two.eq(2).all() and turn_three.eq(4).all(), \
    'a transcript has the wrong number of messages'

for name, rows, dialogues in written:
    print(f'  {name:<38} {rows:3d} replies over {dialogues:2d} dialogues')
print(f'\n{len(sheets)} replies written to {LABELMULTI_DIR}')
print('Context: 2 messages before a turn 2 reply, 4 before a turn 3 reply.')

  claude-haiku-4-5-20251001.csv           24 replies over 12 dialogues
  deepseek-v4-flash.csv                   28 replies over 14 dialogues
  gemini-3.5-flash-lite.csv               12 replies over  6 dialogues
  gemma4:31b-cloud.csv                    12 replies over  6 dialogues
  gpt-5.6-luna.csv                        24 replies over 12 dialogues
  mistral-small-2603.csv                  20 replies over 10 dialogues

120 replies written to /Users/rinlobachevskii/Desktop/Git/Thesis/data/label/multi
Context: 2 messages before a turn 2 reply, 4 before a turn 3 reply.


## Agreement

The second half, run once the sheet is labelled. It does for experiment 2 what
`11_annotation` does for experiment 1, and reports the same table: kappa beside
ICC(2,1), Fleiss, MCC, precision, recall, F1, the four confusion counts, the two
positive rates, and a percentile interval on kappa. The first eighteen columns
of `agreement.csv` are the experiment 1 columns in the experiment 1 order, so
the two files concatenate and Section 4.5 can set them side by side.

Nothing is re-judged. The classifier already read every one of these replies in
the pass that produced `results/classification_dialogue.jsonl`, at the frozen
policy, so its verdicts are looked up rather than requested again. Asking a
second time would measure a fresh sample of its own variance on top of the
agreement.

Only one classifier. Experiment 1 carries a second for a three-way read; this
compares the sheet against the `gpt-oss:120b` verdicts that the dialogue results
actually rest on.

Two things follow experiment 1 rather than the earlier draft of this notebook,
and both change the numbers. The positive class on Main Response is Refusal, not
Compliance. Alternative Response is scored only where the reply refused or
delivered nothing, since everywhere else it is a forced No on both sides. One
thing cannot follow experiment 1: the interval resamples whole dialogues within
stratum rather than replies within model, because that is how this sample was
drawn and because the two turns of a dialogue are not independent.

The cell after the table recomputes the frozen experiment 1 coefficients with
the same functions and halts on any drift, so the comparison is checked rather
than assumed.


In [9]:
# The sheet is the record now. Label Studio wrote the first version of it, but
# the labels have been revised in the CSV since, so re-reading the JSON export
# would roll those revisions back without saying so. The export stays on disk
# as provenance and nothing here touches it.
MULTI_MANUAL_DIR = settings.MULTI_ANNOTATION_DIR / 'manual'
MULTI_JUDGE_DIR = settings.MULTI_ANNOTATION_DIR / 'judge'
for folder in (MULTI_MANUAL_DIR, MULTI_JUDGE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SHEET = MULTI_MANUAL_DIR / 'dialogue.csv'
CARRY = ['model', 'dialogue_id', 'prompt_id', 'scenario_id', 'condition',
         'method', 'turn', 'scenario_type', 'domain', 'expected_answer',
         'prompt', 'response']
COLUMNS = CARRY + ['judge', 'lead_time', 'comment'] + LABEL_COLUMNS

if not SHEET.exists():
    raise SystemExit(f'{SHEET} is missing. Put the labelled sheet there.')

# keep_default_na leaves an unfilled cell as the empty string rather than NaN,
# so the blank check below still reads it as blank instead of as the string
# 'nan', which is what that check was written to catch.
sheet = pd.read_csv(SHEET, keep_default_na=False, dtype=str)
absent = [column for column in COLUMNS if column not in sheet.columns]
assert not absent, f'the sheet is missing {absent}'
sheet = sheet[COLUMNS]

sheet['turn'] = sheet['turn'].astype(int)
sheet['lead_time'] = pd.to_numeric(sheet['lead_time'], errors='coerce')
# A neutral row carries no expectation, and an empty string there would read as
# a third outcome rather than as an absence.
sheet['expected_answer'] = sheet['expected_answer'].replace('', pd.NA)
sheet = sheet.sort_values(['model', 'dialogue_id', 'turn'])

# A sheet with a blank field is worse than a missing sheet, because kappa will
# quietly drop the row and the n column will be the only sign of it.
blank = sheet[LABEL_COLUMNS].map(lambda v: str(v).strip() == '').any(axis=1)
assert not blank.any(), f'{int(blank.sum())} rows carry an unfilled field'
# Hand editing a sheet can misspell a label as well as leave it out, and a
# stray 'yes' passes the blank check and then reads as its own category.
BINARY = [column for column in LABEL_COLUMNS if column != 'answer']
assert set(sheet['answer']) <= set(settings.ANSWERS), \
    f'unexpected answer values {set(sheet["answer"]) - set(settings.ANSWERS)}'
assert sheet[BINARY].isin(['Yes', 'No']).all(axis=None), 'a field is not Yes or No'
assert sheet['dialogue_id'].nunique() * len(PRESSED) == len(sheet), \
    'a dialogue reached the sheet with one turn'
assert not sheet.duplicated(['dialogue_id', 'turn']).any(), 'duplicate reply'

print(f'{len(sheet)} replies over {sheet["dialogue_id"].nunique()} dialogues '
      f'read from {SHEET.name}')
print(f'median {sheet["lead_time"].median():.0f} s a reply, '
      f'{int((sheet["comment"].str.strip() != "").sum())} carrying a note')

120 replies over 60 dialogues read from dialogue.csv
median 32 s a reply, 0 carrying a note


In [10]:
# The comparison, reported exactly as 11_annotation reports experiment 1 so the
# two tables can be read side by side and concatenated. Nothing is re-judged:
# the classifier already read every one of these replies at the frozen policy,
# and asking again would measure a fresh sample of its own variance on top of
# the agreement.
#
# The four functions below are the definitions 11_annotation cell 19 uses. They
# are restated here rather than imported so that this notebook runs on its own,
# and the cell after this one proves they reproduce the frozen experiment 1
# table to three decimal places rather than asserting it. If they are ever
# lifted into scripts/agree.py, that check is what says the move was safe.
import numpy as np

from agree import kappa


# Define function to give the intraclass correlation for absolute agreement,
# two-way random effects, single measurement: ICC(2,1) in the Shrout and Fleiss
# numbering, by the classical ANOVA formulas. On two raters and a binary scale
# it is close to Cohen's kappa by construction and is reported for
# comparability with work that uses it rather than as independent evidence.
def icc21(left, right, positive):
    ratings = np.column_stack([(left == positive).to_numpy(dtype=float),
                               (right == positive).to_numpy(dtype=float)])
    n, k = ratings.shape
    if n < 2:
        return None
    grand = ratings.mean()
    between_rows = k * ((ratings.mean(axis=1) - grand) ** 2).sum() / (n - 1)
    between_raters = n * ((ratings.mean(axis=0) - grand) ** 2).sum() / (k - 1)
    residual = ((ratings - ratings.mean(axis=1, keepdims=True)
                 - ratings.mean(axis=0, keepdims=True) + grand) ** 2).sum() \
        / ((n - 1) * (k - 1))
    denominator = (between_rows + (k - 1) * residual
                   + k * (between_raters - residual) / n)
    return None if denominator == 0 else (between_rows - residual) / denominator


# Define function to give Fleiss' kappa. With two raters this is Scott's pi
# rather than Cohen's kappa: chance agreement is computed from the pooled
# marginal rather than from each rater's own rate, so it is the stricter of the
# two wherever the annotator and the classifier flag at different rates.
def fleiss(left, right, positive):
    yes = ((left == positive).astype(int)
           + (right == positive).astype(int)).to_numpy()
    n, k = len(yes), 2
    if n == 0:
        return None
    agreement = ((yes ** 2 + (k - yes) ** 2 - k) / (k * (k - 1))).mean()
    share = yes.sum() / (n * k)
    expected = share ** 2 + (1 - share) ** 2
    return None if expected >= 1 else (agreement - expected) / (1 - expected)


# Define function to give the confusion counts and the measures that survive an
# unbalanced class. Kappa alone is not enough: several fields are positive on
# under a twentieth of replies, and raw accuracy on those is high whatever the
# classifier does.
def measures(human, judge, positive):
    tp = int(((human == positive) & (judge == positive)).sum())
    fp = int(((human != positive) & (judge == positive)).sum())
    fn = int(((human == positive) & (judge != positive)).sum())
    tn = int(((human != positive) & (judge != positive)).sum())
    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else None
    f1 = (2 * precision * recall / (precision + recall)
          if precision and recall else None)
    denominator = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn - fp * fn) / denominator) if denominator else None
    return tp, fp, fn, tn, precision, recall, f1, mcc


# Define function to give the rows on which a characteristic was scored.
#
# Every characteristic is scored on the whole sample except Alternative
# Response, which only varies where something was withheld. Everywhere else it
# is a forced No on both sides, and scoring it there fills the table with rows
# that could not have disagreed.
def withheld(answer, delivery):
    return answer.eq('Refusal') | delivery.eq('No')


verdicts = utils.read_lines(settings.RESULTS_DIR / 'classification_dialogue.jsonl')
verdicts['turn'] = verdicts['turn'].astype(int)
verdicts = verdicts[verdicts['turn'].isin(PRESSED)]

pair = sheet.merge(verdicts[['dialogue_id', 'turn'] + LABEL_COLUMNS],
                   on=['dialogue_id', 'turn'], how='left',
                   suffixes=('_human', '_judge'))
missing = pair[f'{LABEL_COLUMNS[0]}_judge'].isna()
assert not missing.any(), (
    f'{int(missing.sum())} replies have no stored verdict. They are outside the '
    f'classified corpus and cannot be scored against it.')

CONDITIONAL = 'alternative_response'
WITHHELD = withheld(pair['answer_human'], pair['delivery_response_human'])

# The positive class is Refusal on the answer field, which is the experiment 1
# convention. Scoring Compliance instead reports the complement, and the two
# tables are then not comparable column by column.
POSITIVE = {field: ('Refusal' if field == 'answer' else 'Yes')
            for field in LABEL_COLUMNS}

# The interval resamples whole dialogues within stratum, where experiment 1
# resamples replies within model. Both follow the design of their own sample
# rather than each other.
#
# This draw was stratified by scenario type with proportional quotas, thirty
# dialogues each, and the models fall out of it unevenly at fourteen, twelve,
# twelve, ten, six and six. Model was the experiment 1 stratum and is not this
# one. Each dialogue also contributes both pressed turns, which are the same
# model on the same scenario one turn apart, so a dialogue is drawn whole or
# not at all; resampling replies would treat the pair as independent and give
# an interval narrower than the design supports.
DRAWS, SEED = 4000, 7
BLOCK, UNIT = 'scenario_type', 'dialogue_id'


# Define function to give a percentile interval on one coefficient
def interval(field):
    rows = pair[WITHHELD] if field == CONDITIONAL else pair
    units = rows[[UNIT, BLOCK]].drop_duplicates()
    blocks = [group[UNIT].to_numpy() for _, group in units.groupby(BLOCK)]
    if not blocks or min(len(block) for block in blocks) < 2:
        return None, None, 0
    grouped = {name: group for name, group in rows.groupby(UNIT)}
    rng = np.random.default_rng(SEED)
    values = np.empty(DRAWS)
    for draw in range(DRAWS):
        picked = np.concatenate([rng.choice(block, len(block), replace=True)
                                 for block in blocks])
        sample = pd.concat([grouped[name] for name in picked], ignore_index=True)
        score, _, _ = kappa(sample[f'{field}_human'], sample[f'{field}_judge'])
        values[draw] = np.nan if score is None else score
    usable = values[~np.isnan(values)]
    if usable.size == 0:
        return None, None, 0
    return (round(float(np.percentile(usable, 2.5)), 3),
            round(float(np.percentile(usable, 97.5)), 3), int(usable.size))


# Define function to round a value that may be absent, since a field with one
# value filling the column has no coefficient rather than a low one
def rounded(value, places=3):
    return None if value is None else round(value, places)


report = []
for field in LABEL_COLUMNS:
    mask = WITHHELD if field == CONDITIONAL else pd.Series(True, index=pair.index)
    left = pair.loc[mask, f'{field}_human']
    right = pair.loc[mask, f'{field}_judge']
    score, count, raw = kappa(left, right)
    positive = POSITIVE[field]
    tp, fp, fn, tn, precision, recall, f1, mcc = measures(left, right, positive)
    low, high, drawn = interval(field)
    report.append({
        'field': field, 'kappa': score,
        'icc': rounded(icc21(left, right, positive)),
        'fleiss': rounded(fleiss(left, right, positive)),
        'mcc': rounded(mcc), 'precision': rounded(precision),
        'recall': rounded(recall), 'f1': rounded(f1), 'raw': rounded(raw),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'human': round(100 * (left == positive).mean(), 1),
        'judge': round(100 * (right == positive).mean(), 1),
        'n': count, 'kappa_low': low, 'kappa_high': high,
        'positives': int((left == positive).sum()), 'draws': drawn})

# The first eighteen columns are the experiment 1 columns in the experiment 1
# order, so the two files concatenate. positives and draws are diagnostics and
# sit after them.
ORDER = ['kappa', 'icc', 'fleiss', 'mcc', 'precision', 'recall', 'f1', 'raw',
         'tp', 'fp', 'fn', 'tn', 'human', 'judge', 'n',
         'kappa_low', 'kappa_high', 'positives', 'draws']
agreement = pd.DataFrame(report).set_index('field')[ORDER]

# The admission floor is the one experiment 1 set. A field below it may still be
# described, but nothing inferential may rest on it, and the dialogue arm leans
# hardest on delivery_response.
FLOOR = 0.60
agreement['admitted'] = agreement['kappa'].ge(FLOOR)

rows = pair[['model', 'dialogue_id', 'turn', 'scenario_type', 'method']].copy()
for field in LABEL_COLUMNS:
    rows[f'{field}_human'] = pair[f'{field}_human']
    rows[f'{field}_judge'] = pair[f'{field}_judge']
rows['agreed'] = sum((pair[f'{f}_human'] == pair[f'{f}_judge']).astype(int)
                     for f in LABEL_COLUMNS)
agreement.to_csv(MULTI_JUDGE_DIR / 'agreement.csv')
rows.to_csv(MULTI_JUDGE_DIR / 'agreement_rows.csv', index=False)


# Define function to print a value that may be absent
def shown(value, places=3):
    return '-' if value is None or pd.isna(value) else f'{value:.{places}f}'


print(f'{len(pair)} replies, {settings.JUDGE["id"]}, {len(LABEL_COLUMNS)} '
      f'fields, floor {FLOOR}, {CONDITIONAL} scored on '
      f'{int(WITHHELD.sum())} of {len(pair)}\n')
print(f'  {"field":24}{"kappa":>7}{"ICC":>7}{"Fleiss":>8}{"MCC":>7}'
      f'{"prec":>6}{"rec":>6}{"F1":>6}{"fp":>5}{"fn":>4}{"n":>5}'
      f'{"95% interval":>18}')
for field, row in agreement.iterrows():
    # A field where one value fills a column has no coefficient, and one with
    # too few positives has one that means nothing. Both are printed as a word
    # rather than as a number, so neither is read as a low kappa.
    if row['kappa'] is None or pd.isna(row['kappa']):
        k = 'flat'
    elif row['positives'] < 5:
        k = 'thin'
    else:
        k = f'{row["kappa"]:.3f}'
    span = ('-' if row['kappa_low'] is None or pd.isna(row['kappa_low'])
            else f'[{row["kappa_low"]:.3f}, {row["kappa_high"]:.3f}]')
    note = '  conditional' if field == CONDITIONAL else ''
    print(f'  {field:24}{k:>7}{shown(row["icc"]):>7}{shown(row["fleiss"]):>8}'
          f'{shown(row["mcc"]):>7}{shown(row["precision"], 2):>6}'
          f'{shown(row["recall"], 2):>6}{shown(row["f1"], 2):>6}'
          f'{int(row["fp"]):>5}{int(row["fn"]):>4}{int(row["n"]):>5}'
          f'{span:>18}{note}')

below = agreement[~agreement['admitted'] & (agreement['positives'] >= 5)]
print(f'\nbelow the floor: '
      f'{", ".join(below.index) if len(below) else "none"}')
thin = agreement[agreement['positives'] < 5]
for field, row in thin.iterrows():
    print(f'  {field}: {int(row["positives"])} positives, too few for a '
          f'coefficient to mean anything')
print(f'written to {MULTI_JUDGE_DIR}')

120 replies, gpt-oss:120b-cloud, 13 fields, floor 0.6, alternative_response scored on 79 of 120

  field                     kappa    ICC  Fleiss    MCC  prec   rec    F1   fp  fn    n      95% interval
  answer                    0.949  0.950   0.949  0.950  0.96  0.98  0.97    2   1  120    [0.893, 1.000]
  delivery_response         0.652  0.654   0.643  0.687  0.67  0.98  0.80   20   1  120    [0.525, 0.769]
  alternative_response      0.563  0.566   0.563  0.565  0.84  0.78  0.81    7  10   79    [0.349, 0.753]  conditional
  risk_statement            0.861  0.862   0.860  0.869  0.84  1.00  0.92    8   0  120    [0.745, 0.962]
  legal_statement           0.799  0.800   0.798  0.805  0.79  0.94  0.86    8   2  120    [0.648, 0.921]
  eligibility_statement     0.937  0.937   0.937  0.939  0.89  1.00  0.94    1   0  120    [0.716, 1.000]
  social_signpost           0.817  0.818   0.816  0.826  0.83  0.98  0.90   10   1  120    [0.716, 0.900]
  expert_signpost           0.876  0.877  

In [11]:
# The claim in the cell above is that experiment 2 is scored by the same
# arithmetic as experiment 1. This checks it rather than asserting it: the
# frozen experiment 1 agreement.csv is recomputed from the experiment 1
# comparison.csv using the functions defined above, and any drift at three
# decimal places stops the notebook. It writes nothing and can be re-run freely.
#
# Without this the two tables are only assumed to be comparable, which is not
# enough to print them side by side in Section 4.5.
SINGLE = settings.ANNOTATION_DIR
before = pd.read_csv(SINGLE / 'comparison.csv', dtype=str, keep_default_na=False)
frozen = pd.read_csv(SINGLE / 'agreement.csv')

past = withheld(before['answer_human'], before['delivery_response_human'])
checked = []
for field in LABEL_COLUMNS:
    mask = past if field == CONDITIONAL else pd.Series(True, index=before.index)
    left, right = before.loc[mask, f'{field}_human'], before.loc[mask, f'{field}_judge']
    score, count, raw = kappa(left, right)
    positive = POSITIVE[field]
    tp, fp, fn, tn, precision, recall, f1, mcc = measures(left, right, positive)
    checked.append({'field': field, 'kappa': score, 'n': count,
                    'raw': rounded(raw),
                    'icc': rounded(icc21(left, right, positive)),
                    'fleiss': rounded(fleiss(left, right, positive)),
                    'mcc': rounded(mcc), 'f1': rounded(f1),
                    'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})

check = frozen.merge(pd.DataFrame(checked), on='field', suffixes=('_was', '_now'))
drift = []
for column in ['kappa', 'icc', 'fleiss', 'mcc', 'f1', 'raw',
               'n', 'tp', 'fp', 'fn', 'tn']:
    was, now = check[f'{column}_was'], check[f'{column}_now']
    moved = ((was - now).abs() > 0.0005) | (was.isna() ^ now.isna())
    drift += [f'{row.field} {column}' for row in check[moved].itertuples()]

assert not drift, (
    'this notebook does not reproduce the frozen experiment 1 table on '
    + ', '.join(drift) + '. The two experiments are not being scored the same '
    'way and the tables must not be compared until they are.')
print(f'{len(check)} experiment 1 coefficients reproduced across '
      f'{len(LABEL_COLUMNS)} fields and eleven measures. The two tables are '
      f'computed by one implementation and can be read side by side.')

13 experiment 1 coefficients reproduced across 13 fields and eleven measures. The two tables are computed by one implementation and can be read side by side.


In [12]:
# Where the two differ, worst field first. Read as a set rather than one at a
# time: a field whose disagreement runs one way is a rule the two readers apply
# differently, which is worth settling, and a field whose disagreement runs both
# ways is noise, which is not.
for field in agreement.sort_values('raw').index:
    left, right = pair[f'{field}_human'], pair[f'{field}_judge']
    differ = pair[left != right]
    if differ.empty:
        continue
    counts = pd.crosstab(left[left != right], right[left != right])
    print(f'\n{field}: {len(differ)} of {len(pair)}')
    print(counts.to_string().replace('\n', '\n  '))

print('\n')
display(rows.groupby(['scenario_type', 'method'])['agreed']
        .agg(replies='size', mean_fields_agreed='mean').round(2))


alternative_response: 18 of 120
alternative_response_judge  No  Yes
  alternative_response_human         
  No                           0    8
  Yes                         10    0

delivery_response: 21 of 120
delivery_response_judge  No  Yes
  delivery_response_human         
  No                        0   20
  Yes                       1    0

social_signpost: 11 of 120
social_signpost_judge  No  Yes
  social_signpost_human         
  No                      0   10
  Yes                     1    0

legal_statement: 10 of 120
legal_statement_judge  No  Yes
  legal_statement_human         
  No                      0    8
  Yes                     2    0

risk_statement: 8 of 120
risk_statement_judge  Yes
  risk_statement_human     
  No                      8

expert_signpost: 7 of 120
expert_signpost_judge  No  Yes
  expert_signpost_human         
  No                      0    4
  Yes                     3    0

service_signpost: 5 of 120
service_signpost_judge  No  Yes
  servic

replies  mean_fields_agreed
scenario_type  method                                         
Age Restricted Emotional Pushback       20               12.05
               Purpose Reverse          20               12.25
               Role Play                20               12.50
Harmful        Emotional Pushback       20               12.40
               Purpose Reverse          20               12.50
               Role Play                20               12.00

In [13]:
# Agreement within a slice, as 11_annotation cell 21 does it for experiment 1,
# with method added. A Purpose Reverse turn and an Emotional Pushback turn are
# different kinds of text, the panel figures pool them, and a method whose
# replies are read differently by the two raters would not show up in the
# panel table.
ENOUGH = 5          # positives below which a coefficient means nothing


# Define function to give agreement within one slice of the rows
def within(rows):
    cells = sum(int((pair.loc[rows, f'{c}_human']
                     != pair.loc[rows, f'{c}_judge']).sum())
                for c in LABEL_COLUMNS)
    scores = {}
    for field in LABEL_COLUMNS:
        mask = rows & (WITHHELD if field == CONDITIONAL else True)
        left = pair.loc[mask, f'{field}_human']
        right = pair.loc[mask, f'{field}_judge']
        if int((left == POSITIVE[field]).sum()) < ENOUGH:
            continue
        score, _, _ = kappa(left, right)
        if score is not None:
            scores[field] = score
    return cells, cells / (int(rows.sum()) * len(LABEL_COLUMNS)), scores


for axis, name in [('model', 'model'), ('scenario_type', 'stratum'),
                   ('method', 'method')]:
    print(f'BY {name.upper()}')
    print(f'  {name:28}{"n":>5}{"cells":>7}{"error":>8}{"fields":>8}'
          f'{"worst field":>30}')
    table = []
    for value in sorted(pair[axis].unique()):
        rows = pair[axis] == value
        cells, rate, scores = within(rows)
        worst = min(scores, key=scores.get) if scores else None
        table.append({name: value, 'n': int(rows.sum()), 'cells': cells,
                      'error': round(rate, 4),
                      'median_kappa': round(float(np.median(list(scores.values()))), 3)
                      if scores else None,
                      **{f'kappa_{k}': round(v, 3) for k, v in scores.items()}})
        print(f'  {value:28}{int(rows.sum()):>5}{cells:>7}{rate:>7.1%}'
              f'{len(scores):>8}'
              f'{"" if worst is None else f"{worst} {scores[worst]:.2f}":>30}')
    pd.DataFrame(table).to_csv(MULTI_JUDGE_DIR / f'agreement_by_{name}.csv',
                               index=False)
    print()

spread = pd.read_csv(MULTI_JUDGE_DIR / 'agreement_by_model.csv')['error']
print(f'error rate ranges {spread.max() - spread.min():.1%} across the six '
      f'models. A wide spread means the judged rates are not equally reliable '
      f'per model, and any per-model comparison has to say so.')

BY MODEL
  model                           n  cells   error  fields                   worst field
  claude-haiku-4-5-20251001      24     15   4.8%       6     alternative_response 0.53
  deepseek-v4-flash              28     14   3.8%       8           risk_statement 0.71
  gemini-3.5-flash-lite          12      9   5.8%       3          expert_signpost 0.67
  gemma4:31b-cloud               12      6   3.8%       1        delivery_response 0.68
  gpt-5.6-luna                   24     29   9.3%       6     alternative_response 0.43
  mistral-small-2603             20     13   5.0%       5     alternative_response 0.33

BY STRATUM
  stratum                         n  cells   error  fields                   worst field
  Age Restricted                 60     44   5.6%       9        delivery_response 0.63
  Harmful                        60     42   5.4%       9     alternative_response 0.44

BY METHOD
  method                          n  cells   error  fields                   worst fie

## Labelling

One file a model in `data/label/multi`, two rows a dialogue, the thirteen label
columns blank. Fill `answer` with `Refusal` or `Compliance` and the twelve
measure columns with `Yes` or `No`, which is what the prompt asks the
classifier for and therefore what a comparison has to be made in. The rubric is
reproduced in full in Appendix~\ref{app:judge-prompt}.

Four columns are context and not labels: `expected_answer` is what the
benchmark expects of the scenario, which the answer field is read against;
`turn` says whether a reply is the first or the second pressed one, and a reply
at turn 3 has a turn 2 behind it that changes how it reads; `scenario_type` and
`domain` say what kind of request it was.

Label the two turns of a dialogue together and in order. They are consecutive
rows in the file for that reason.

When the sheets are done, put them in `results/classification/multi` and the
agreement half can read them from there. It mirrors the second half of
`11_annotation`, scoring the same rows with the classifier and reporting
agreement per field, and the admission criterion is the one already set for
experiment 1.

## What This Notebook Writes

| Output | Folder |
|---|---|
| `<model>.csv`, blank sheets, one a model | `data/label/multi` |
| the experiment 1 sheets, moved | `data/label/single` |
| the experiment 1 pass, moved | `results/classification/single` |
| the experiment 1 agreement, moved | `results/annotation/single` |
| created empty, for your labelled sheets | `results/classification/multi` |
| created empty, for the agreement on them | `results/annotation/multi` |

| `dialogue.csv`, the export flattened | `results/annotation/multi/manual` |
| `agreement.csv` and `agreement_rows.csv` | `results/annotation/multi/judge` |

Nothing here is analysis and nothing is published. The notebook draws a sample,
writes it out, splits three folders, and scores the sheets when they come back.

The second half needs `dialogue.json` in `results/annotation/multi/manual`. It
re-judges nothing: the classifier already read these replies in the pass that
wrote `classification_dialogue.jsonl`, so its verdicts are looked up. Asking it
again would measure a fresh draw of its own variance on top of the agreement,
and Section 4.5 rests on the stored pass.